# Module 36 — Approximate Nearest Neighbors with FAISS

> **Track 3 · Yandex Big Tech & Search**  
> **Difficulty**: Advanced  
> **Runtime**: ~10 minutes  
> **Key Libraries**: FAISS, NumPy, Plotly  
> **Prerequisite**: Module 29 (Two-Tower Retrieval)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Embeddings](#3-synthetic-embeddings)
4. [FAISS Index Types](#4-faiss-index-types)
5. [Search Performance](#5-search-performance)
6. [Recall vs Speed Tradeoff](#6-recall-vs-speed-tradeoff)
7. [Visualization](#7-visualization)
8. [Key Business Takeaway](#8-key-business-takeaway)

---
## §1 · Business Context
    
### Why FAISS for similarity search?

FAISS enables **fast similarity search** at scale:
- **Approximate nearest neighbors**: Trade accuracy for speed.
- **GPU acceleration**: Leverage GPUs for massive speedups.
- **Production proven**: Used by Facebook, Yandex, and others.

**Applications at Yandex**:
1. **Semantic search**: Find similar documents by embedding.
2. **Recommendation**: Retrieve similar items quickly.
3. **Duplicate detection**: Find near-duplicate content.

**Key index types**:
- **Flat**: Exact search, slow but accurate.
- **IVF**: Inverted file index for faster search.
- **HNSW**: Hierarchical navigable small world graphs.

In this notebook we will:
1. Generate synthetic embeddings (1M vectors).
2. Build different FAISS indexes.
3. Benchmark recall vs speed tradeoffs.
4. Visualize search performance.

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── FAISS ────────────────────────────────────────────────────────
import faiss

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  FAISS version: {faiss.__version__}")

---
## §3 · Synthetic Embeddings

In [ ]:
# Generate synthetic embeddings
N_VECTORS = 1_000_000
DIMENSION = 128

# Generate random embeddings
embeddings = np.random.randn(N_VECTORS, DIMENSION).astype('float32')

# Normalize embeddings
faiss.normalize_L2(embeddings)

print(f"✓ Generated {N_VECTORS:,} embeddings")
print(f"  Dimension: {DIMENSION}")
print(f"  Memory: {embeddings.nbytes / 1e9:.2f} GB")

---
## §4 · FAISS Index Types

In [ ]:
# Create different FAISS indexes
# 1. Flat index (exact search)
print("Building Flat index...")
t0 = time.time()
index_flat = faiss.IndexFlatIP(DIMENSION)  # Inner product (cosine for normalized)
index_flat.add(embeddings)
flat_build_time = time.time() - t0
print(f"  Built in {flat_build_time:.2f}s")

# 2. IVF index (inverted file)
print("\nBuilding IVF index...")
t0 = time.time()
nlist = 1000  # Number of clusters
quantizer = faiss.IndexFlatIP(DIMENSION)
index_ivf = faiss.IndexIVFFlat(quantizer, DIMENSION, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(embeddings)
index_ivf.add(embeddings)
ivf_build_time = time.time() - t0
print(f"  Built in {ivf_build_time:.2f}s")

# 3. HNSW index
print("\nBuilding HNSW index...")
t0 = time.time()
M = 32  # Number of connections
index_hnsw = faiss.IndexHNSWFlat(DIMENSION, M)
index_hnsw.add(embeddings)
hnsw_build_time = time.time() - t0
print(f"  Built in {hnsw_build_time:.2f}s")

print(f"\n✓ All indexes built")

---
## §5 · Search Performance

In [ ]:
# Generate query vectors
N_QUERIES = 1000
K = 10  # Top-K results

queries = np.random.randn(N_QUERIES, DIMENSION).astype('float32')
faiss.normalize_L2(queries)

# Search with each index
def search_index(index, queries, k):
    t0 = time.time()
    distances, indices = index.search(queries, k)
    search_time = time.time() - t0
    return distances, indices, search_time

# Flat search
flat_dists, flat_indices, flat_time = search_index(index_flat, queries, K)
print(f"Flat search: {flat_time:.4f}s ({N_QUERIES/flat_time:.0f} queries/sec)")

# IVF search
index_ivf.nprobe = 10  # Number of clusters to search
ivf_dists, ivf_indices, ivf_time = search_index(index_ivf, queries, K)
print(f"IVF search: {ivf_time:.4f}s ({N_QUERIES/ivf_time:.0f} queries/sec)")

# HNSW search
hnsw_dists, hnsw_indices, hnsw_time = search_index(index_hnsw, queries, K)
print(f"HNSW search: {hnsw_time:.4f}s ({N_QUERIES/hnsw_time:.0f} queries/sec)")

---
## §6 · Recall vs Speed Tradeoff

In [ ]:
# Compute recall (compare against flat search)
def compute_recall(predicted, ground_truth, k):
    """Compute recall@k."""
    recalls = []
    for i in range(len(predicted)):
        pred_set = set(predicted[i][:k])
        gt_set = set(ground_truth[i][:k])
        recall = len(pred_set & gt_set) / len(gt_set)
        recalls.append(recall)
    return np.mean(recalls)

# Compute recall for IVF and HNSW
ivf_recall = compute_recall(ivf_indices, flat_indices, K)
hnsw_recall = compute_recall(hnsw_indices, flat_indices, K)

print("=" * 70)
print("RECALL VS SPEED TRADEOFF")
print("=" * 70)
print(f"\nFlat (exact):")
print(f"  Recall@{K}: 1.0000")
print(f"  Speed: {N_QUERIES/flat_time:.0f} queries/sec")
print(f"\nIVF (approximate):")
print(f"  Recall@{K}: {ivf_recall:.4f}")
print(f"  Speed: {N_QUERIES/ivf_time:.0f} queries/sec")
print(f"  Speedup: {flat_time/ivf_time:.1f}x")
print(f"\nHNSW (approximate):")
print(f"  Recall@{K}: {hnsw_recall:.4f}")
print(f"  Speed: {N_QUERIES/hnsw_time:.0f} queries/sec")
print(f"  Speedup: {flat_time/hnsw_time:.1f}x")

---
## §7 · Visualization

In [ ]:
# Visualize performance
fig = make_subplots(rows=1, cols=2, subplot_titles=['Search Speed', 'Recall vs Speed'])

# Search speed
methods = ['Flat', 'IVF', 'HNSW']
speeds = [N_QUERIES/flat_time, N_QUERIES/ivf_time, N_QUERIES/hnsw_time]

fig.add_trace(go.Bar(
    x=methods,
    y=speeds,
    name='Queries/sec',
    marker_color=['#636EFA', '#EF553B', '#00CC96']
), row=1, col=1)

# Recall vs Speed
recalls = [1.0, ivf_recall, hnsw_recall]

fig.add_trace(go.Scatter(
    x=speeds,
    y=recalls,
    mode='markers+text',
    text=methods,
    textposition='top center',
    name='Methods',
    marker=dict(size=15)
), row=1, col=2)

fig.update_layout(height=400, showlegend=True)
fig.show()

print("💡 Key insights:")
print("   - HNSW provides best recall-speed tradeoff")
print("   - IVF is faster but may miss some results")
print("   - Flat is accurate but slow for large datasets")

---
## §8 · Key Business Takeaway

> ### 🎯 Action Items for Search & Recommendation Teams
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Similarity search, recommendation, deduplication |
> | **Index type** | HNSW for best recall-speed, IVF for memory efficiency |
> | **Dimension** | 128-512 dimensions typical for embeddings |
> | **Scale** | Millions to billions of vectors |
> | **Latency** | < 10ms for real-time search |
>
> ### 💡 Production Considerations
>
> 1. **Index selection guide**:
>    | Dataset Size | Index Type | Memory | Speed |
>    |--------------|------------|--------|-------|
>    | < 1M | Flat | Low | Slow |
>    | 1M-100M | IVF | Medium | Fast |
>    | > 100M | HNSW | High | Very Fast |
>
> 2. **GPU acceleration**:
>    - Use `faiss.GpuIndexFlatIP` for GPU support
>    - 10-100x speedup over CPU
>    - Essential for real-time applications
>
> 3. **Production tips**:
>    - Pre-compute embeddings offline
>    - Update index periodically (daily/weekly)
>    - Monitor recall with ground truth samples
>
> ### 🔑 Key Takeaways
>
> 1. **FAISS enables fast similarity search** at billion scale.
> 2. **HNSW provides best recall-speed tradeoff** for most use cases.
> 3. **GPU acceleration** is essential for real-time applications.
> 4. **Pre-compute embeddings** and update index periodically.
> 5. **Monitor recall** to ensure search quality.